# Find polymorphisms across multiple gorilla individuals


For a set of target loci (ampliconic gene families) with known sequence from the reference, this notebook finds their copies in each individual assembly, counts copies, extracts the regions, and recovers the coding sequence with miniprot.

Several X-ampliconic families have Y paralogs, and the individual assemblies are contig-level rather than chromosome-resolved, so a naive BLAT/BLAST search would also pick up Y (and autosomal) contigs. To avoid that, we first align each assembly to the reference X with `wfmash` to identify which contigs carry X content, and restrict the search to those. (run wfmash.sh script)


## Pipeline stages
0. Config — paths, individuals, target loci, thresholds.
1. X-linked contigs — `wfmash` each assembly to reference X, keep X-mapping contigs, subset the FASTA.
2. Reference queries — pull the spliced mRNA (BLAT query) and protein (miniprot query) for each target locus from the GFF.
3. Search — BLAT reference mRNAs against each individual's X-contigs, filter hits.
4. Copy number — cluster hits into discrete loci, one count per gene per individual.
5. Extract regions — pull each copy's interval (±flank) to FASTA.
6. miniprot — align the reference protein to each copy to get its CDS / flag pseudogenes.
7. Summary — copy-number table across individuals.

Requires `wfmash`, `gffread`, `blat`/`pblat`, `miniprot`, `samtools`, `bedtools`, `seqkit` on PATH (or set absolute paths in `TOOLS` below).

## Stage 0 — Configuration

Edit this cell; everything after it is generic.


In [ ]:
from pathlib import Path
import subprocess, shlex, re
import pandas as pd

# ----- species/individuals ---------------------------------------------------
SPECIES     = "" # chimp, gorilla, human

INDIVIDUALS = [""] # this is the list of individuals             

# ----- reference ---------------------------------------------------------------
REF_FASTA = Path("/path/to/reference/genome/of/that/species/{species}_X.fasta")
REF_GFF   = Path("/path/to/reference/gff/of/that/species/{species}_X.gff)


# ----- individual genome path template -----------------------------------------
GENOME_TMPL = ("/path/to/haploid/assembly/of/individual/provided/above/{ind}/haploid_assembly/{ind}_haploid.fa")

# ----- target gene families ----------------------------------------------------
# Each family maps to the list of its member loci (gene symbols / LOC ids as they
# appear in the GFF Name= field). All members are used as BLAT queries, then hits
# are deduped *within a family* so each genomic copy is counted once.
GENE_FAMILIES = { # fill these in for the specific species 
    "VCX":  [],
    "SSX": [], 
    "HSFX": [], 
    "GAGE": [], 
    "CSAG": [],
    "CT45": [],
    "MAGEB6": ["MAGEB6", "MAGEB6B"], # example
}


# flat (family, locus) pairs used throughout
FAMILY_LOCI = [(fam, loc) for fam, loci in GENE_FAMILIES.items() for loc in loci]

# ----- thresholds (tune these) -------------------------------------------------
MIN_X_ALN  = 20_000     # bp: min cumulative alignment to call a contig X-linked
MIN_IDENT  = 0.90       # min fraction identity for a BLAT hit to be kept
MIN_QCOV   = 0.50       # min fraction of the query mRNA that must align
MERGE_DIST = 1_000      # bp: hits on same contig+strand within this = one copy
FLANK      = 2_000      # bp: flank added when extracting each copy region

# ----- output ------------------------------------------------------------------
OUTDIR = Path(f"/pop_data/{SPECIES}")
OUTDIR.mkdir(parents=True, exist_ok=True)

#----- tools / resources -------------------------------------------------------
TOOLS = {  # set absolute paths if not on PATH
    "wfmash":   "/miniforge3/envs/wfmash/bin/wfmash",
    "gffread": "gffread", "blat": "blat", "pblat": "~/miniforge3/envs/pblat/bin/pblat",
    "miniprot": "miniprot", "samtools": "samtools", "bedtools": "~/miniforge3/envs/pblat/bin/bedtools",
    "seqkit": "seqkit", 
    "iqtree": "iqtree3"
}
THREADS = 10

def run(cmd):
    """Echo and run a shell command; raise if it fails."""
    print("›", cmd)
    subprocess.run(cmd, shell=True, check=True, text=True)

print(f"{len(INDIVIDUALS)} individuals, {len(GENE_FAMILIES)} families "
      f"({len(FAMILY_LOCI)} loci) -> {OUTDIR}")

## Stage 1 — Identify X-linked contigs

Align each individual assembly (query) to the reference X (target) with `wfmash`, then keep query contigs whose cumulative alignment to X exceeds `MIN_X_ALN`. If `REF_Y_FASTA` is set, also align to Y and drop contigs that look more Y than X. Subset the assembly FASTA to the kept contigs so later steps only see X content.


In [ ]:
# Stage 1 was precomputed in parallel by wfmash.sh (SLURM array).
# point to the per-individual X-contig FASTAs it produced.
x_contig_fastas = {}
for ind in INDIVIDUALS:
    xfa = OUTDIR / ind / f"{ind}_Xcontigs.fa"
    assert xfa.exists(), f"missing {xfa} — did the sbatch job finish for {ind}?"
    fai = Path(str(xfa) + ".fai")
    if not fai.exists():
        run(f"{TOOLS['samtools']} faidx {xfa}")        # needed by Stage 5
    n = sum(1 for _ in open(OUTDIR / ind / f"{ind}_Xcontigs.txt"))
    x_contig_fastas[ind] = xfa
    print(f"  {ind}: {n} X-linked contigs (precomputed)")

## Stage 2 — Reference queries (mRNA + protein) for every family member

Extracts every member locus of every family. For each locus we match the `Name=` field to a `gene`, pick the longest-CDS isoform, and use `gffread` to write:

- `loci_mrna.fa` — spliced exon sequence, used as the BLAT query
- `loci_protein.faa` — translated CDS, used as the miniprot query

FASTA headers are renamed to `family|locus` so every downstream step knows which family a query belongs to (to be able dedupe within a family later). `loci_map.tsv` (family, locus, gene_id, transcript_id) is written for reference.

Loci not found in the GFF are printed as a warning. 


In [ ]:
def _attrs(field):
    d = {}
    for kv in field.split(";"):
        if "=" in kv:
            k, v = kv.split("=", 1)
            d[k] = v
    return d

def select_reference_loci():
    genes_by_name = {}        # SYMBOL(upper) -> [gene_id, ...]
    mrnas         = {}        # mrna_id -> (parent_gene_id, attrs)
    cds_len       = {}        # mrna_id -> total CDS length
    children      = {}        # mrna_id -> list of raw GFF lines (exon/CDS)
    gene_line     = {}        # gene_id -> raw gene line

    with open(REF_GFF) as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            if len(f) < 9:
                continue
            ftype, a = f[2], _attrs(f[8])
            if ftype == "gene":
                gid = a.get("ID", "")
                gene_line[gid] = line
                name = a.get("Name", a.get("gene", "")).upper()
                genes_by_name.setdefault(name, []).append(gid)
            elif ftype in ("mRNA", "transcript"):
                rid = a.get("ID", "")
                mrnas[rid] = (a.get("Parent", ""), a)
            elif ftype in ("exon", "CDS"):
                rid = a.get("Parent", "")
                children.setdefault(rid, []).append(line)
                if ftype == "CDS":
                    cds_len[rid] = cds_len.get(rid, 0) + (int(f[4]) - int(f[3]) + 1)

    chosen, missing = [], []          # chosen: (family, locus, gene_id, mrna_id)
    for family, locus in FAMILY_LOCI:
        gids = genes_by_name.get(locus.upper(), [])
        if not gids:
            missing.append(f"{family}:{locus}"); continue
        cand = [rid for rid, (parent, _) in mrnas.items() if parent in gids]
        if not cand:
            missing.append(f"{family}:{locus}"); continue
        best = max(cand, key=lambda rid: cds_len.get(rid, 0))   # longest CDS
        chosen.append((family, locus, mrnas[best][0], best))

    if missing:
        print("not found in GFF:", ", ".join(missing))

    # write filtered GFF containing only the chosen transcripts (+ gene & children)
    sub_gff = OUTDIR / "loci.gff"
    chosen_rids = {rid for _, _, _, rid in chosen}
    with open(sub_gff, "w") as out:
        out.write("##gff-version 3\n")
        seen_gene = set()
        for family, locus, gid, rid in chosen:
            if gid in gene_line and gid not in seen_gene:
                out.write(gene_line[gid]); seen_gene.add(gid)
        with open(REF_GFF) as fh:
            for line in fh:
                if line.startswith("#"):
                    continue
                f = line.rstrip("\n").split("\t")
                if len(f) < 9:
                    continue
                a = _attrs(f[8])
                if f[2] in ("mRNA", "transcript") and a.get("ID", "") in chosen_rids:
                    out.write(line)
                elif f[2] in ("exon", "CDS") and a.get("Parent", "") in chosen_rids:
                    out.write(line)

    # mapping used to rename FASTA headers: norm(transcript_id) -> "family|locus"
    def _norm(t):
        return t[4:] if t.startswith("rna-") else t
    rid2name = {_norm(rid): f"{family}|{locus}" for family, locus, gid, rid in chosen}

    with open(OUTDIR / "loci_map.tsv", "w") as m:
        m.write("family\tlocus\tgene_id\ttranscript_id\n")
        for family, locus, gid, rid in chosen:
            m.write(f"{family}\t{locus}\t{gid}\t{rid}\n")

    print(f"selected {len(chosen)} loci across {len({c[0] for c in chosen})} families "
          f"-> {sub_gff}")
    return sub_gff, rid2name

def rename_fasta_headers(fa, rid2name):
    """Rewrite each FASTA header to 'family|locus' using the transcript-id map."""
    def _norm(t):
        return t[4:] if t.startswith("rna-") else t
    out = []
    for line in open(fa):
        if line.startswith(">"):
            tok = line[1:].split()[0]
            name = rid2name.get(_norm(tok))
            if name is None:
                print("could not map header", tok, "in", fa.name); name = tok
            out.append(">" + name + "\n")
        else:
            out.append(line)
    Path(fa).write_text("".join(out))

sub_gff, rid2name = select_reference_loci()
mrna_fa   = OUTDIR / "loci_mrna.fa"
prot_fa   = OUTDIR / "loci_protein.faa"
run(f"{TOOLS['gffread']} {sub_gff} -g {shlex.quote(str(REF_FASTA))} "
    f"-w {mrna_fa} -y {prot_fa}")
rename_fasta_headers(mrna_fa, rid2name)
rename_fasta_headers(prot_fa, rid2name)
run(f"grep -c '>' {mrna_fa} {prot_fa}")


## Stage 3 — BLAT search + hit filtering

BLAT the reference mRNAs against each individual's X-contigs. BLAT splices the mRNA across exons, so each PSL row is a candidate copy. Keep rows passing `MIN_IDENT` and `MIN_QCOV`.

PSL columns (no header): matches, misMatches, repMatches, nCount, qNumInsert, qBaseInsert, tNumInsert, tBaseInsert, strand, qName, qSize, qStart, qEnd, tName, tSize, tStart, tEnd, blockCount, blockSizes, qStarts, tStarts.


In [ ]:
# ================= Stage 3: BLAT search + hit filtering =================
PSL_COLS = ["matches","misMatches","repMatches","nCount","qNumInsert","qBaseInsert",
            "tNumInsert","tBaseInsert","strand","qName","qSize","qStart","qEnd",
            "tName","tSize","tStart","tEnd","blockCount","blockSizes","qStarts","tStarts"]
_INT = {"matches","misMatches","repMatches","nCount","qNumInsert","qBaseInsert",
        "tNumInsert","tBaseInsert","qSize","qStart","qEnd","tSize","tStart","tEnd","blockCount"}

SPAN_FACTOR = 3.0     # drop a hit if its target span > SPAN_FACTOR x the family's median span
                      # (removes chimeric BLAT alignments that chain across tandem-array units)

def blat_search(ind, xfa):
    psl = OUTDIR / ind / f"{ind}_loci.psl"
    tool = TOOLS["pblat"]; thr = f"-threads={THREADS} "
    try:
        run(f"{tool} {thr}-t=dna -q=dna -minIdentity={int(MIN_IDENT*100)} "
            f"{xfa} {mrna_fa} {psl}")
    except Exception:
        run(f"{TOOLS['blat']} -t=dna -q=dna -minIdentity={int(MIN_IDENT*100)} "
            f"{xfa} {mrna_fa} {psl}")
    return psl

def parse_psl(psl):
    rows = []
    with open(psl) as fh:
        for line in fh:
            f = line.rstrip("\n").split("\t")
            if len(f) < 21 or not f[0].isdigit():
                continue
            r = dict(zip(PSL_COLS, f))
            for k in _INT:
                r[k] = int(r[k])
            aligned = r["matches"] + r["misMatches"] + r["repMatches"]
            r["identity"] = (r["matches"] + r["repMatches"]) / aligned if aligned else 0
            r["qcov"]     = (r["qEnd"] - r["qStart"]) / r["qSize"] if r["qSize"] else 0
            r["tspan"]    = r["tEnd"] - r["tStart"]          # genomic span of the hit
            rows.append(r)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    # identity + query-coverage filters
    return df[(df.identity >= MIN_IDENT) & (df.qcov >= MIN_QCOV)].reset_index(drop=True)

# build identity/qcov-filtered hits per individual
hits = {}
for ind in INDIVIDUALS:
    df = parse_psl(blat_search(ind, x_contig_fastas[ind]))
    if not df.empty:
        df["individual"] = ind
        df["family"] = df.qName.str.split("|").str[0]
        df["locus"]  = df.qName.str.split("|").str[1]
    hits[ind] = df

# adaptive per-family span filter (removes chimeric array-spanning hits)
allhits = pd.concat([d for d in hits.values() if not d.empty], ignore_index=True) \
          if any(not d.empty for d in hits.values()) else pd.DataFrame(columns=["family","tspan"])
fam_median_span = allhits.groupby("family").tspan.median()      # typical single-copy span, robust

def _drop_chimeric(df):
    if df.empty:
        return df
    cap = df.family.map(fam_median_span) * SPAN_FACTOR
    return df[df.tspan <= cap].reset_index(drop=True)

for ind in INDIVIDUALS:
    before = len(hits[ind])
    hits[ind] = _drop_chimeric(hits[ind])
    print(f"  {ind}: {len(hits[ind])} hits kept "
          f"({before - len(hits[ind])} chimeric array-spanning hits dropped)")


## Stage 4 — Collapse hits into copies, deduped within each family

 Within a family, the same genomic copy is typically hit by several member loci, & each copy can give several split PSL rows. Group by family × contig × strand, sort by target start, and merge any intervals within `MERGE_DIST` into one copy. Overlapping hits from different members of the same family collapse to a single counted copy.

For each copy keep the best identity/coverage seen, which member locus gave the best hit (`best_locus`), and the full set of members that hit it (`loci`).


In [ ]:
def cluster_copies(df):
    """Merge hits within a family into discrete copies (the dedup step)."""
    if df.empty:
        return pd.DataFrame()
    out = []
    keys = ["family", "tName", "strand"]
    for (family, contig, strand), g in df.sort_values(keys + ["tStart"]).groupby(keys):
        cur = None
        for _, row in g.iterrows():
            s, e = row.tStart, row.tEnd
            if cur and s <= cur["tEnd"] + MERGE_DIST:          # same copy -> merge
                cur["tEnd"]      = max(cur["tEnd"], e)
                cur["best_qcov"] = max(cur["best_qcov"], row.qcov)
                cur["loci"].add(row.locus)
                cur["n_hits"]   += 1
                if row.identity > cur["best_id"]:
                    cur["best_id"], cur["best_locus"] = row.identity, row.locus
            else:                                              # start a new copy
                if cur:
                    out.append(cur)
                cur = dict(individual=row.individual, family=family, contig=contig,
                           strand=strand, tStart=s, tEnd=e,
                           best_id=row.identity, best_qcov=row.qcov,
                           best_locus=row.locus, loci={row.locus}, n_hits=1)
        if cur:
            out.append(cur)
    res = pd.DataFrame(out)
    if not res.empty:
        res["loci"] = res["loci"].apply(lambda s: ",".join(sorted(s)))  # members that hit
    return res

copies = pd.concat([cluster_copies(df) for df in hits.values() if not df.empty],
                   ignore_index=True)
copies["copy_id"] = (copies.individual + "__" + copies.family + "__" +
                     copies.contig + "_" + copies.tStart.astype(str) +
                     "-" + copies.tEnd.astype(str) + "_" + copies.strand)
copies.to_csv(OUTDIR / "copies.csv", index=False)
copies.head()

## Stage 5 — Extract each copy region (±flank)

Write a BED per individual and pull the sequence with `bedtools getfasta`. Coordinates are clipped to contig ends using the `.fai`.


In [ ]:
def contig_sizes(fai):
    return {l.split("\t")[0]: int(l.split("\t")[1]) for l in open(fai)}

copy_fastas = {}
for ind in INDIVIDUALS:
    sub = copies[copies.individual == ind]
    if sub.empty:
        continue
    xfa  = x_contig_fastas[ind]
    sizes = contig_sizes(f"{xfa}.fai")
    bed  = OUTDIR / ind / f"{ind}_copies.bed"
    with open(bed, "w") as out:
        for _, r in sub.iterrows():
            start = max(0, r.tStart - FLANK)
            end   = min(sizes.get(r.contig, r.tEnd + FLANK), r.tEnd + FLANK)
            out.write(f"{r.contig}\t{start}\t{end}\t{r.copy_id}\t0\t{r.strand}\n")
    cfa = OUTDIR / ind / f"{ind}_copies.fa"
    run(f"{TOOLS['bedtools']} getfasta -s -name -fi {xfa} -bed {bed} -fo {cfa}")
    copy_fastas[ind] = cfa
    print(f"  {ind}: extracted {len(sub)} copy regions -> {cfa.name}")

## Stage 6 — miniprot per copy (in isolation): recover the CDS

A single genome-wide miniprot run caps the number of alignments kept per protein, so in large ampliconic families some real copies get out-competed and left unannotated even though they'd align fine on their own. To avoid that,  annotate each BLAT copy in isolation: extract just that copy's region (+flank), run miniprot there with only that family's proteins, and take the best alignment overlapping the copy (so a neighbour in the flank can't be grabbed by mistake). Since each region contains essentially one target, miniprot places it reliably.

For each annotated copy record identity/frameshift/stop info and write its CDS and protein to combined files keyed by `copy_id`.

In [ ]:
import subprocess

def _run_quiet(cmd):
    subprocess.run(cmd, shell=True, check=True, text=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# one protein FASTA per family (headers are 'family|locus'), built once
_fam_prot = {}
def family_proteins(family):
    if family not in _fam_prot:
        out, keep = OUTDIR / f"_prot_{family}.faa", False
        with open(prot_fa) as fh, open(out, "w") as o:
            for line in fh:
                if line.startswith(">"):
                    keep = (line[1:].split("|")[0] == family)
                if keep:
                    o.write(line)
        _fam_prot[family] = out
    return _fam_prot[family]

def annotate_copy_isolated(row):
    """miniprot on just this copy's region; return (annotation, region_fa, gff, mrna_id)."""
    ind, family, contig = row.individual, row.family, row.contig
    cstart, cend = int(row.tStart), int(row.tEnd)
    work = OUTDIR / ind
    xfa  = x_contig_fastas[ind]

    rstart = max(1, cstart - FLANK)
    region = f"{contig}:{rstart}-{cend + FLANK}"
    rfa = work / f"_iso_{row.copy_id}.fa"
    gff = work / f"_iso_{row.copy_id}.gff"
    # extract region, rename its header to a simple 'region' to avoid ':' issues
    _run_quiet(f"{TOOLS['samtools']} faidx {xfa} {region} | sed '1s/.*/>region/' > {rfa}")
    _run_quiet(f"{TOOLS['miniprot']} --gff {rfa} {family_proteins(family)} > {gff}")

    # copy interval in region coordinates; keep only mRNAs overlapping THIS copy
    c0, c1 = cstart - rstart + 1, cend - rstart + 1
    best = None
    for line in open(gff):
        if line.startswith("#"):
            continue
        f = line.rstrip("\n").split("\t")
        if len(f) < 9 or f[2] != "mRNA":
            continue
        ms, me = int(f[3]), int(f[4])
        if me < c0 or ms > c1:
            continue
        a = _attrs(f[8])
        ident = pd.to_numeric(a.get("Identity", ""), errors="coerce")
        if best is None or (pd.notna(ident) and ident > best["mp_identity"]):
            best = dict(mp_id=a.get("ID", ""), mp_identity=ident,
                        mp_frameshifts=int(a.get("Frameshift", "0") or 0),
                        mp_stops=a.get("StopCodon", ""))
    return best, rfa, gff

def append_cds(row, rfa, gff, mrna_id, cds_out, prot_out):
    """Extract this copy's CDS/protein and append, with the header set to copy_id."""
    work = OUTDIR / row.individual
    sel  = work / f"_iso_{row.copy_id}.sel.gff"
    with open(gff) as g, open(sel, "w") as o:
        o.write("##gff-version 3\n")
        for line in g:
            if line.startswith("#"):
                continue
            f = line.split("\t")
            if len(f) < 9:
                continue
            a = _attrs(f[8])
            if (f[2] == "mRNA" and a.get("ID", "") == mrna_id) or \
               (f[2] in ("CDS", "stop_codon") and a.get("Parent", "") == mrna_id):
                o.write(line)
    tcds, tprot = work / f"_iso_{row.copy_id}.cds", work / f"_iso_{row.copy_id}.prot"
    _run_quiet(f"{TOOLS['gffread']} {sel} -g {rfa} -x {tcds} -y {tprot}")
    for src, dst in [(tcds, cds_out), (tprot, prot_out)]:
        with open(src) as s, open(dst, "a") as d:
            for line in s:
                d.write(f">{row.copy_id}\n" if line.startswith(">") else line)
    for p in (sel, tcds, tprot):
        p.unlink(missing_ok=True)

# ---- annotate every copy, collect CDS keyed by copy_id ----
for col in ["mp_id", "mp_identity", "mp_frameshifts", "mp_stops"]:
    copies[col] = pd.NA
cds_out, prot_out = OUTDIR / "copies_cds.fa", OUTDIR / "copies_protein.faa"
open(cds_out, "w").close(); open(prot_out, "w").close()

for n, (i, row) in enumerate(copies.iterrows(), 1):
    best, rfa, gff = annotate_copy_isolated(row)
    if best is not None:
        for k in ["mp_id", "mp_identity", "mp_frameshifts", "mp_stops"]:
            copies.at[i, k] = best[k]
        append_cds(row, rfa, gff, best["mp_id"], cds_out, prot_out)
    rfa.unlink(missing_ok=True); gff.unlink(missing_ok=True)
    if n % 50 == 0 or n == len(copies):
        print(f"  annotated {n}/{len(copies)} copies")

copies["intact"] = (copies.mp_id.notna() &
                    (copies.mp_frameshifts.fillna(1).astype(float) == 0))
copies.to_csv(OUTDIR / "copies.csv", index=False)
print(f"{copies.mp_id.notna().sum()}/{len(copies)} copies annotated; "
      f"NA: {copies.mp_id.isna().sum()}.  CDS -> {cds_out.name}")
copies.head()

In [ ]:
# Stage 6b: add the reference genome as an individual
import subprocess, shlex

REF_IND = "reference"          # the label the reference gets in every table and tree

def _run_quiet(cmd):
    subprocess.run(cmd, shell=True, check=True, text=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def read_fasta(path):
    d, name, seq = {}, None, []
    for line in open(path):
        line = line.rstrip()
        if line.startswith(">"):
            if name: d[name] = "".join(seq)
            name, seq = line[1:].split()[0], []
        else: seq.append(line)
    if name: d[name] = "".join(seq)
    return d

def _norm(t): return t[4:] if t.startswith("rna-") else t

sub_gff  = OUTDIR / "loci.gff"          # built in Stage 2
cds_path = OUTDIR / "copies_cds.fa"     # built in Stage 6

# 1) reference CDS straight from the annotation
ref_cds_raw = OUTDIR / "_reference_cds_raw.fa"
_run_quiet(f"{TOOLS['gffread']} {sub_gff} -g {shlex.quote(str(REF_FASTA))} -x {ref_cds_raw}")
raw = read_fasta(ref_cds_raw)

# 2) transcript -> (family, locus) from loci_map.tsv
tx2fl = {}
with open(OUTDIR / "loci_map.tsv") as fh:
    next(fh)
    for line in fh:
        fam, loc, gid, tx = line.rstrip("\n").split("\t")
        tx2fl[_norm(tx)] = (fam, loc)

# 3) transcript -> (contig, start, end, strand) from the sub-GFF mRNA lines
tx2coord = {}
for line in open(sub_gff):
    if line.startswith("#"): continue
    f = line.rstrip("\n").split("\t")
    if len(f) < 9 or f[2] not in ("mRNA", "transcript"): continue
    a = dict(kv.split("=", 1) for kv in f[8].split(";") if "=" in kv)
    tx2coord[_norm(a.get("ID", ""))] = (f[0], int(f[3]), int(f[4]), f[6])

# 4) build reference "copies" and their CDS
ref_rows, to_add = [], []
for header, seq in raw.items():
    key = _norm(header.split()[0])
    if key not in tx2fl or key not in tx2coord:
        continue
    fam, loc = tx2fl[key]
    contig, start, end, strand = tx2coord[key]
    cid = f"{REF_IND}__{fam}__{contig}_{start}-{end}_{strand}"
    to_add.append((cid, seq))
    ref_rows.append(dict(individual=REF_IND, family=fam, contig=contig, strand=strand,
                         tStart=start, tEnd=end, best_id=1.0, best_qcov=1.0,
                         best_locus=loc, loci=loc, n_hits=1, copy_id=cid,
                         mp_id=f"ref:{loc}", mp_identity=1.0,
                         mp_frameshifts=0, mp_stops="", intact=True))

# 5) append reference CDS to copies_cds.fa (guarded so it can't double-append)
if "reference__" not in open(cds_path).read():
    with open(cds_path, "a") as o:
        for cid, seq in to_add:
            o.write(f">{cid}\n{seq}\n")

# 6) merge reference rows into `copies` (idempotent: drop any old reference rows first)
copies = copies[copies.individual != REF_IND].copy()
copies = pd.concat([copies, pd.DataFrame(ref_rows)], ignore_index=True)

print(f"added {len(ref_rows)} reference loci as individual '{REF_IND}'")
display(copies[copies.individual == REF_IND].groupby("family").size().rename("reference_copies"))

## Stage 7 — Copy-number summary

`family × individual` copy counts (deduped within family in Stage 4), plus a second table counting only copies miniprot judged intact (no frameshift).


In [ ]:
copy_number = (copies.groupby(["family", "individual"]).size()
                     .unstack(fill_value=0).sort_index())
copy_number.to_csv(OUTDIR / "copy_number_summary.csv")

intact_number = (copies[copies.intact].groupby(["family", "individual"]).size()
                       .unstack(fill_value=0)
                       .reindex(index=copy_number.index, columns=copy_number.columns,
                                fill_value=0))
intact_number.to_csv(OUTDIR / "intact_copy_number_summary.csv")

print("Total copies per family/individual:"); display(copy_number)
print("Intact (no-frameshift) copies:");       display(intact_number)

## Stage 8 — synonymous vs nonsynonymous differences, within vs between individuals
 

In [ ]:
# Stage : within-individual syn/nonsyn counts
import subprocess
from itertools import permutations, combinations
from Bio.Seq import Seq

def _run_quiet(cmd):
    subprocess.run(cmd, shell=True, check=True, text=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def read_fasta(path):
    d, name, seq = {}, None, []
    for line in open(path):
        line = line.rstrip()
        if line.startswith(">"):
            if name: d[name] = "".join(seq)
            name, seq = line[1:].split()[0], []
        else: seq.append(line)
    if name: d[name] = "".join(seq)
    return d

# single source of truth for copy names: {individual}_{family}_{n}
def make_copy_names():
    s = copies.sort_values(["family", "individual", "contig", "tStart"]).copy()
    s["tip"] = s.individual + "_" + s.family + "_" + \
               (s.groupby(["family", "individual"]).cumcount() + 1).astype(str)
    return dict(zip(s.copy_id, s.tip))

COPY_NAME = make_copy_names()          # copy_id -> tip name (used in 8 AND 9)

# Nei-Gojobori counting
BASES = "TCAG"
AA = {a+b+c: str(Seq(a+b+c).translate()) for a in BASES for b in BASES for c in BASES}

def codon_sites(c):
    aa = AA.get(c)
    if aa is None or aa == '*': return None
    syn = sum(1 for i in range(3) for b in BASES
              if b != c[i] and AA[c[:i]+b+c[i+1:]] == aa)
    return syn/3.0, 3.0 - syn/3.0

def codon_diffs(c1, c2):
    if c1 == c2: return 0.0, 0.0
    a1, a2 = AA.get(c1), AA.get(c2)                    # None for gap/frameshift/ambiguous
    if a1 in (None, '*') or a2 in (None, '*'): return None
    dp = [i for i in range(3) if c1[i] != c2[i]]
    if len(dp) == 1:
        return (0.0, 1.0) if a1 != a2 else (1.0, 0.0)
    paths = []
    for order in permutations(dp):
        cur, s, n, ok = c1, 0.0, 0.0, True
        for i in order:
            nxt = cur[:i] + c2[i] + cur[i+1:]
            an = AA.get(nxt)
            if an in (None, '*'): ok = False; break
            s, n = (s+1, n) if AA[cur] == an else (s, n+1)
            cur = nxt
        if ok: paths.append((s, n))
    if not paths: return None
    return (sum(p[0] for p in paths)/len(paths),
            sum(p[1] for p in paths)/len(paths))

def pair_stats(a, b):
    a, b = a.upper(), b.upper()
    Sd = Nd = S = N = 0.0
    for k in range(0, min(len(a), len(b)), 3):
        ca, cb = a[k:k+3], b[k:k+3]
        sa, sb, d = codon_sites(ca), codon_sites(cb), codon_diffs(ca, cb)
        if sa is None or sb is None or d is None:      # skips '-', '!', stops, ambiguous
            continue
        S += (sa[0]+sb[0])/2; N += (sa[1]+sb[1])/2
        Sd += d[0]; Nd += d[1]
    return Sd, Nd, S, N

# MACSE alignment per (family, individual)
ALN_DIR = OUTDIR / "family_alignments"; ALN_DIR.mkdir(exist_ok=True)

def macse_align(seqs, tag):
    fin = ALN_DIR / f"{tag}.fa"; nt = ALN_DIR / f"{tag}_NT.fa"; aa = ALN_DIR / f"{tag}_AA.fa"
    with open(fin, "w") as o:
        for name, s in seqs.items():
            o.write(f">{name}\n{s}\n")
    macse = TOOLS.get("macse", "macse")
    _run_quiet(f"{macse} -prog alignSequences -seq {fin} -out_NT {nt} -out_AA {aa}")
    _run_quiet(f"{macse} -prog refineAlignment -align {nt} -out_NT {nt} -out_AA {aa}")
    return read_fasta(nt)

# run over every (family, individual); set intact_only=True to restrict
intact_only = True
cds = read_fasta(OUTDIR / "copies_cds.fa")
intact_map = copies.set_index("copy_id")["intact"].to_dict()
fs_map     = copies.set_index("copy_id")["mp_frameshifts"].to_dict()

sum_rows, pair_rows, copy_rows = [], [], []
src = copies[copies.intact] if intact_only else copies

for (family, ind), grp in src.groupby(["family", "individual"]):
    ids = [c for c in grp.copy_id if c in cds]
    if len(ids) < 2:
        continue
    aln_name = {f"c{i}": cid for i, cid in enumerate(ids)}      # MACSE-safe -> copy_id
    aln = macse_align({sn: cds[cid] for sn, cid in aln_name.items()}, f"{family}__{ind}")

    for sn, cid in aln_name.items():
        seq = aln.get(sn, "")
        copy_rows.append(dict(family=family, individual=ind,
                              copy=COPY_NAME[cid], copy_id=cid,
                              intact=intact_map.get(cid),
                              miniprot_frameshifts=fs_map.get(cid),
                              macse_frameshift_sites=seq.count("!"),
                              aligned_bases=len(seq.replace("-", "").replace("!", ""))))

    intact_in = sum(1 for cid in ids if intact_map.get(cid))
    sum_Sd = sum_Nd = 0.0
    print(f"\n=== {family} in {ind}: {len(ids)} copies ({intact_in} intact) ===")
    for a, b in combinations(sorted(aln), 2):
        Sd, Nd, S, N = pair_stats(aln[a], aln[b])
        sum_Sd += Sd; sum_Nd += Nd
        nameA, nameB = COPY_NAME[aln_name[a]], COPY_NAME[aln_name[b]]
        both = intact_map.get(aln_name[a]) and intact_map.get(aln_name[b])
        print(f"  {nameA}  vs  {nameB}:  syn={Sd:.1f}  nonsyn={Nd:.1f}"
              f"{'' if both else '   [non-intact]'}")
        pair_rows.append(dict(family=family, individual=ind, copyA=nameA, copyB=nameB,
                              intactA=intact_map.get(aln_name[a]),
                              intactB=intact_map.get(aln_name[b]),
                              Sd=Sd, Nd=Nd, S=round(S, 1), N=round(N, 1)))
    n_pairs = len(ids) * (len(ids) - 1) // 2
    print(f"  --> SUM syn={sum_Sd:.1f} nonsyn={sum_Nd:.1f}  |  "
          f"per-pair syn={sum_Sd/n_pairs:.2f} nonsyn={sum_Nd/n_pairs:.2f}")
    sum_rows.append(dict(family=family, individual=ind, n_copies=len(ids),
                         n_intact=intact_in, n_pairs=n_pairs,
                         sum_Sd=sum_Sd, sum_Nd=sum_Nd))

within_individual = pd.DataFrame(sum_rows)
within_individual["mean_Sd_per_pair"] = within_individual.sum_Sd / within_individual.n_pairs
within_individual["mean_Nd_per_pair"] = within_individual.sum_Nd / within_individual.n_pairs
pairs_within = pd.DataFrame(pair_rows)
copies_diag  = pd.DataFrame(copy_rows)
within_individual.to_csv(OUTDIR / "within_individual_syn_nonsyn.csv", index=False)
pairs_within.to_csv(OUTDIR / "within_individual_pairs.csv", index=False)
copies_diag.to_csv(OUTDIR / "within_individual_copies.csv", index=False)

print("\n\nPer-individual sums + normalized:"); display(within_individual.sort_values(["family","individual"]))
print("Per-copy diagnostics:");                display(copies_diag.sort_values(["family","individual"]))

In [ ]:
# Stage 8c: syn|nonsyn tally tables (family = columns, individual = rows)
wi = within_individual.copy()

def pivot_combined(df, s_col, n_col, fmt):
    df = df.copy()
    df["cell"] = df[s_col].map(fmt.format) + " | " + df[n_col].map(fmt.format)
    return df.pivot(index="individual", columns="family", values="cell").fillna("-")

# 1) totals: summed synonymous | nonsynonymous over all within-individual pairs
totals = pivot_combined(wi, "sum_Sd", "sum_Nd", "{:.1f}")

# 2) normalized over the number of copies in that individual
wi["sd_per_copy"] = wi.sum_Sd / wi.n_copies
wi["nd_per_copy"] = wi.sum_Nd / wi.n_copies
per_copy = pivot_combined(wi, "sd_per_copy", "nd_per_copy", "{:.2f}")

totals.to_csv(OUTDIR / "syn_nonsyn_totals.csv")
per_copy.to_csv(OUTDIR / "syn_nonsyn_per_copy.csv")

print("Total  synonymous | nonsynonymous  (within-individual, per family):")
display(totals)
print("\nNormalized per copy  (syn|nonsyn ÷ n_copies in that individual):")
display(per_copy)

In [ ]:
# Stage 8d: pN / pS tables (differences / sites), family cols x individual rows
# pool over within-individual pairs: sum diffs and sum sites separately, then divide
agg = (pairs_within.groupby(["family", "individual"])
       .agg(Sd=("Sd", "sum"), Nd=("Nd", "sum"),
            S=("S", "sum"),  N=("N", "sum")).reset_index())
agg["pS"] = agg.Sd / agg.S          # synonymous differences per synonymous site
agg["pN"] = agg.Nd / agg.N          # nonsynonymous differences per nonsynonymous site

def pivot_combined(df, a, b, fmt):
    df = df.copy()
    df["cell"] = df[a].map(fmt.format) + " | " + df[b].map(fmt.format)
    return df.pivot(index="individual", columns="family", values="cell").fillna("-")

# pS | pN  (this is the site-normalized, length-fair version of your totals)
pspn = pivot_combined(agg, "pS", "pN", "{:.4f}")

# and the ratio pN/pS, the constraint / selection summary
agg["pN_pS"] = agg.pN / agg.pS
ratio = (agg.pivot(index="individual", columns="family", values="pN_pS")
         .round(3).fillna("-"))

pspn.to_csv(OUTDIR / "pS_pN_by_family.csv")
ratio.to_csv(OUTDIR / "pNpS_by_family.csv")

print("pS | pN  (differences ÷ sites, pooled over within-individual pairs):")
display(pspn)
print("\npN / pS :")
display(ratio)

# Make per family a tree for the copies of multiple individuals

In [ ]:
# Stage 9: per-family tree across individuals, coloured by individual
import subprocess, matplotlib.pyplot as plt
from Bio import Phylo

TREE_DIR = OUTDIR / "family_trees"; TREE_DIR.mkdir(exist_ok=True)

def _run(cmd):
    print("›", cmd); subprocess.run(cmd, shell=True, check=True, text=True)

def read_fasta(path):
    d, name, seq = {}, None, []
    for line in open(path):
        line = line.rstrip()
        if line.startswith(">"):
            if name: d[name] = "".join(seq)
            name, seq = line[1:].split()[0], []
        else: seq.append(line)
    if name: d[name] = "".join(seq)
    return d

def build_family_tree(family, intact_only=True, threads=THREADS):
    sub = copies[copies.family == family].copy()
    if intact_only:
        sub = sub[sub.intact]
    # deterministic naming: {individual}_{family}_{n}
    sub = sub.sort_values(["individual", "contig", "tStart"])
    sub["tip"] = sub.individual + "_" + family + "_" + \
                 (sub.groupby("individual").cumcount() + 1).astype(str)
    name_map = dict(zip(sub.copy_id, sub.tip))     # copy_id -> tip
    ind_of   = dict(zip(sub.tip, sub.individual))  # tip -> individual

    cds = read_fasta(OUTDIR / "copies_cds.fa")
    fin, n = TREE_DIR / f"{family}.fa", 0
    with open(fin, "w") as o:
        for cid, tip in name_map.items():
            if cid in cds:                          # NA copies have no CDS -> excluded
                o.write(f">{tip}\n{cds[cid]}\n"); n += 1
    print(f"{family}: {n} copies with a CDS across {sub.individual.nunique()} individuals")
    if n < 3:
        print("  too few sequences for a tree"); return None, ind_of

    # pooled MACSE codon alignment
    nt, aa = TREE_DIR / f"{family}_NT.fa", TREE_DIR / f"{family}_AA.fa"
    macse = TOOLS.get("macse", "macse")
    _run(f"{macse} -prog alignSequences -seq {fin} -out_NT {nt} -out_AA {aa}")
    _run(f"{macse} -prog refineAlignment -align {nt} -out_NT {nt} -out_AA {aa}")

    # clean for tree building: MACSE frameshift '!' -> N (missing)
    aln, clean = read_fasta(nt), TREE_DIR / f"{family}_NT_clean.fa"
    with open(clean, "w") as o:
        for name, s in aln.items():
            o.write(f">{name}\n{s.replace('!', 'N')}\n")

    # IQ-TREE: model selection + ultrafast bootstrap
    _run(f"{TOOLS.get('iqtree', 'iqtree3')} -s {clean} -m MFP -B 1000 "
         f"-T AUTO --redo --keep-ident --prefix {TREE_DIR / family}")
    return TREE_DIR / f"{family}.treefile", ind_of

def plot_tree(treefile, ind_of, family):
    tree = Phylo.read(str(treefile), "newick")
    tree.ladderize()
    inds = sorted(set(ind_of.values()))
    cmap = plt.get_cmap("tab20" if len(inds) > 10 else "tab10")
    colors = {ind: cmap(i) for i, ind in enumerate(inds)}
    lc = {tip: colors[ind] for tip, ind in ind_of.items()}

    fig, ax = plt.subplots(figsize=(10, max(4, 0.22 * len(ind_of))))
    Phylo.draw(tree, axes=ax, do_show=False,
               label_colors=lambda n: lc.get((n or "").replace(" ", "_"), "black"))
    handles = [plt.Line2D([0], [0], color=colors[i], lw=4, label=i) for i in inds]
    ax.legend(handles=handles, title="individual", loc="lower right", fontsize=8)
    ax.set_title(f"{family}: all copies across individuals"); ax.set_xlabel("substitutions/site")
    plt.tight_layout(); plt.savefig(TREE_DIR / f"{family}_tree.png", dpi=150); plt.show()

# --- run for VCX (loop over families if you like) ---
#tf, ind_of = build_family_tree("VCX")
#if tf: plot_tree(tf, ind_of, "VCX")
family_trees = {}
for fam in sorted(copies.family.unique()):
    print(f"\n########## {fam} ##########")
    try:
        tf, io = build_family_tree(fam)
        if tf:
            plot_tree(tf, io, fam)
            family_trees[fam] = tf
    except Exception as e:
        print(f"  skipped {fam}: {e}")

print("\nTrees built for:", ", ".join(family_trees))

In [ ]:
inv = {v: k for k, v in COPY_NAME.items()}          # tip name -> copy_id
lookup = copies.assign(tip=copies.copy_id.map(COPY_NAME))[
    ["tip", "individual", "family", "contig", "tStart", "tEnd", "strand",
     "best_locus", "loci", "mp_id", "intact"]]
lookup.to_csv(OUTDIR / "copy_name_lookup.csv", index=False)
lookup.head()

In [ ]:
# Final: rename + restyle from saved files (
# Reads copies.csv and the Stage 9 .treefile outputs. Writes renamed PDFs + a name map.
import matplotlib.pyplot as plt
from Bio import Phylo

SPECIES_LABEL = "Gorilla"        # <-- EDIT per species: Human / Chimpanzee / Gorilla
TREE_DIR = OUTDIR / "family_trees"
PDF_DIR  = OUTDIR / "final_trees"; PDF_DIR.mkdir(exist_ok=True)

cp = pd.read_csv(OUTDIR / "copies.csv")

# --- 1. individual -> Species_n  (reference keeps its name) ---
reals = sorted(i for i in cp.individual.unique() if i != "reference")
IND_LABEL = {ind: f"{SPECIES_LABEL}_{k}" for k, ind in enumerate(reals, 1)}
IND_LABEL["reference"] = "reference"

ind_map = pd.DataFrame({"original": list(IND_LABEL), "label": list(IND_LABEL.values())})
ind_map["species"] = SPECIES_LABEL
ind_map.to_csv(OUTDIR / "individual_name_map.csv", index=False)
print("Individual name map:"); display(ind_map)

# --- 2. rebuild BOTH the old tip names (as Stage 9 wrote them) and the new ones ---
s = cp.sort_values(["family", "individual", "contig", "tStart"]).copy()
n = s.groupby(["family", "individual"]).cumcount() + 1
s["old_tip"] = s.individual + "_" + s.family + "_" + n.astype(str)          # what's in the treefile
s["new_tip"] = s.individual.map(IND_LABEL) + "_copy" + n.astype(str)       # what we want
OLD2NEW  = dict(zip(s.old_tip, s.new_tip))
TIP2IND  = dict(zip(s.new_tip, s.individual.map(IND_LABEL)))

# --- 3. redraw each family's tree from its .treefile ---
def redraw(family):
    tf = TREE_DIR / f"{family}.treefile"
    if not tf.exists():
        print(f"  no treefile for {family}"); return
    tree = Phylo.read(str(tf), "newick"); tree.ladderize()

    for term in tree.get_terminals():                 # rename tips in place
        if term.name in OLD2NEW:
            term.name = OLD2NEW[term.name]

    tips  = [t.name for t in tree.get_terminals()]
    ind_of = {t: TIP2IND[t] for t in tips if t in TIP2IND}
    inds  = sorted(set(ind_of.values()))
    cmap  = plt.get_cmap("tab20" if len(inds) > 10 else "tab10")
    colors = {ind: cmap(i) for i, ind in enumerate(inds)}
    lc = {t: colors[i] for t, i in ind_of.items()}

    fig, ax = plt.subplots(figsize=(10, max(4, 0.22 * len(tips))))
    Phylo.draw(tree, axes=ax, do_show=False,
               label_colors=lambda nm: lc.get((nm or "").replace(" ", "_"), "black"),
               label_func=lambda c: c.name if c.is_terminal() else "",   # hide bootstraps
               show_confidence=False)
    ax.set_ylabel(""); ax.set_yticks([])                                  # no y-axis
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)                                # x-axis only
    ax.set_xlabel("substitutions/site")
    ax.set_title(family)                                                  # family in title only
    handles = [plt.Line2D([0], [0], color=colors[i], lw=4, label=i) for i in inds]
    #ax.legend(handles=handles, frameon=False, loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(PDF_DIR / f"{family}_tree.pdf", format="pdf", bbox_inches="tight")
    plt.show()
    print(f"  saved {family}_tree.pdf")

for fam in sorted(cp.family.unique()):
    redraw(fam)

# --- 4. renamed tables ---
cp["individual"] = cp.individual.map(IND_LABEL)
copy_number = (cp[cp.intact.astype(str).str.lower().isin(["true","1"])]
               .groupby(["family","individual"]).size()
               .unstack(fill_value=0).T)                                  # rows=individual
copy_number.to_csv(OUTDIR / "final_copy_number.csv")
print("\nRenamed copy number (intact):"); display(copy_number)